In [2]:
/* 
  MicroGrad was written by Andrej Karpathy in Python, accompanying a youtube
  video explaining how to construct a MultiLayerPerceptron. In order to 
  understand it better, I have translated the Python code to Javascript. 

  You can find Andrej Karpathy's video at
  
    https://www.youtube.com/watch?v=VMj-3S1tku0
  
  and the python source code at
  
    https://github.com/karpathy/micrograd
  

  The code below defines 4 classes:

  - The Value class, which encapsulates a numerical value, and operations 
    that can be performed on it, including a backward operation, which traces 
    the calculations that led to this value back to their origins, while
    determining their influence on this result.

  - The Neuron, which consists of a number of weights w, and a bias b. A 
    Neuron can be called, like a function, using its call() method. When 
    supplied with an array of input values, the call function adds the inputs
    multiplied by their respective weights, and the bias, returning this 
    value's .relu().

  - The Layer is a collection of Neurons. When the Layer is .call()ed, it 
    passes its arguments to the neurons, returning in turn, their return
    values.

  - The MultiLayerPerceptron (MLP) in turn is a collection of layers. Calling
    the MLP returns a what the layers return.

  Training the MLP works by initializing it to random weights, that is Layers
  with Neurons with random Values, calculating the loss between the intended
  and the produced ("predicted") y-values, then calling backward() on the loss
  to produce each weight's gradient, (the influence on the result), and 
  finally nudging the weights (also called the parameters) away from the 
  gradient. Since the gradients of the weights are their influence on the 
  loss, adjusting the parameters away from the loss makes the overall
  calculation more accurate, which is our goal.

  This is my understanding of MicroGrad, please feel to comtact me if I got 
  something wrong. My email address is mooreolith@gmail.com.
*/


undefined

In [4]:
/*
  MicroGrad by Andrej Karpathy
  
  Translated to JavaScript by Joshua Moore
*/

// A Value encapsulates a number, as well as the operators and inputs 
// that led to it.
class Value {
  constructor(data, children=[], op='', label=''){
    this.data = data;
    this.prev = new Set(children);
    this.op = op;
    this.label = label;
    this.grad = 0;
    this._backward = () => {};
  }

  toString(){
    return `Value ${this.label + ' '}(data=${this.data.toFixed(3)} grad=${this.grad.toFixed(3)})`;
  }

  add(other){
    other = other instanceof Value ? other : new Value(other);
    const out = new Value(this.data + other.data, [this, other], '+');
    out._backward = () => {
      this.grad += out.grad;
      other.grad += out.grad;
    };
    
    return out;
  }

  mul(other){
    other = other instanceof Value ? other : new Value(other)
    const out = new Value(this.data * other.data, [this, other], '*');
    out._backward = () => {
      this.grad += other.data * out.grad;
      other.grad += this.data * out.grad;
    }
    
    return out;
  }

  pow(other){
    console.assert(`Argument other must be a Number.`, other instanceof Number);
    const out = new Value(this.data**other, [this], `**${other}`);
    out._backward = () => {
      this.grad += (other * this.data**(other-1)) * out.grad;
    }

    return out;
  }

  relu(){
    const out = new Value(this.data < 0 ? 0 : this.data, [this], 'ReLu');
    out._backward = () => {
      this.grad += (out.data > 0 ? 1 : 0) * out.grad;
    }

    return out;
  }

  backward(){
    const topo = [];
    const visited = new Set();
    const buildTopo = (v) => {
      if(!visited.has(v)){
        visited.add(v);
        for(let child of v.prev){
          buildTopo(child);
        }
        topo.push(v);
      }
    };
    buildTopo(this);

    this.grad = 1.0;
    for(let i = topo.length -1; i>=0; i--){
      topo[i]._backward();
    }
    
    return topo;
  }

  neg(){
    return this.mul(-1);
  }

  sub(other){
    other = other instanceof Value ? other : new Value(other);
    return this.add(other.neg());
  }

  div(other){
    return this.mul(other.pow(-1));
  }

  tanh(){
    const x = this.data;
    const t = (Math.exp(2*x) - 1)/(Math.exp(2*x) + 1);
    const out = new Value(t, [this], 'tanh');
    out._backward = () => {
      this.grad += (1- t**2) * out.grad;
    }

    return out;
  }

  exp(){
    const x = this.data;
    const out = new Value(Math.exp(x), [this], 'exp');
    out._backward = () => {
      this.grad += out.data * out.grad;
    }

    return out;
  }

  trace(){
    const nodes = new Set(), edges = new Set();
    const build = (v) => {
      nodes.add(v);
      for(let child of v.prev){
        edges.add([child, v]);
        build(child);
      }
    }
    build(root);
    return nodes, edges;
  }
}

// A Neuron of Values
class Neuron {
  constructor(nin, nonlin=true){
    this.w = Array(nin).fill(0).map(() => new Value((Math.random() * 2) - 1));
    this.b = new Value(0);
    this.nonlin = nonlin;
  }

  call(x){

    // act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)

    const act = this.w
      .reduce((sum, wi, i) => sum.add(wi.mul(x[i])), new Value(0))
      .add(this.b);

    return this.nonlin ? act.relu() : act;
  }

  parameters(){
    return this.w.concat([this.b]);
  }

  toString(){
    return `${this.nonlin ? 'ReLu' : 'Linear'} Neuron(${this.w.length})`;
  }
}

// A Layer of Neurons
class Layer {
  constructor(nin, nout, nonlin){
    this.neurons = Array(nout).fill(0).map(() => new Neuron(nin, nonlin));
  }

  call(x){
    const outs = this.neurons.map(n => n.call(x));
    return outs.length == 1 ? outs[0] : outs;
  }

  parameters(){
    const params = [];
    for(let neuron of this.neurons){
      for(let p of neuron.parameters()){
        params.push(p);
      }
    }
    
    return params;
  }

  toString(){
    return `Layer of [{${this.neurons.map(n => n.toString()).join(', ')}}]`
  }
}

// MultiLayerPerceptron
class MLP {
  constructor(nin, nouts){
    const sz = [nin].concat(...nouts);
    this.layers = Array(nouts.length).fill(0).map((_, i) => new Layer(sz[i], sz[i+1], i < nouts.length - 1))
  }

  call(x){
    for(let layer of this.layers){
      x = layer.call(x);
    }
    
    return x;
  }

  parameters(){
    let params = [];
    for(let layer of this.layers){
      for(let p of layer.parameters()){
        params.push(p);
      }
    }

    return params;
  }

  toString(){
    return `MLP of [{${this.layers.map(l => l.toString()).join(', ')}}]`
  }

  train(xs, ys, ROUNDS=100, NABLA=0.01){
    let predYs, loss;    
    for(let i=0; i<ROUNDS; i++){
      // predict
      predYs = xs.map(x => this.call(x));
      loss = ys
        .map((ty, i) => predYs[i].sub(ty).pow(2))
        .reduce((sum, l) => sum.add(l), new Value(0));

      // reset gradients
      // mlp.parameters().forEach(p => p.grad = 0.0);
      let params = this.parameters();
      for(let p of params){
        p.grad = 0;
      }

      // calculate gradients
      loss.backward();

      // tweak parameters
      params = this.parameters();
      for(let p of params){
        p.data += -NABLA * p.grad;
      }
    }
  }

  predict(xs){
    return xs.map(x => mlp.call(x));
  }

  loss(ys, predYs){
    return ys
      .map((ty, i) => predYs[i].sub(ty).pow(2))
      .reduce((sum, l) => sum.add(l), new Value(0));
  }
}

/*
  Training
*/
// Define training data
let xs = [
  [2, 3, -1],
  [3, -1, .5],
  [.5, 1, 1],
  [1, 1, -1]
];

let ys = [
  1, 
  -1, 
  -1, 
  1
];

console.log(`Xs: ${JSON.stringify(xs)}`);
console.log(`Ys: ${JSON.stringify(ys)}`)

// Instantiate MLP (to random values)
let mlp = new MLP(3, [4, 4, 1]);
let predYs, loss;
console.log(`\n`);
console.log(`MLP(3, [4, 4, 1]), randomly initialized`)

// Sneak a peek at what the MLP with untrained, random weights returns
predYs = mlp.predict(xs);
loss = mlp.loss(ys, predYs);
console.log(`\n`);
console.log(`Initial Prediction: [${[...predYs.map(y => y.data.toFixed(3))].join(", ")}] `);
console.log(`Initial Loss: ${loss.data.toFixed(3)}`);

// Train the MLP's weights with xs and ys defined above
let ROUNDS = 100, RATE = 0.01;
mlp.train(xs, ys, ROUNDS, RATE);
console.log("\n");
console.log(`Training done. Rounds: ${ROUNDS}, Learning Rate: ${RATE}`)

// See the trained MLP in action
predYs = mlp.predict(xs);
loss = mlp.loss(ys, predYs);
console.log(`\n`);
console.log(`Trained Prediction: [${[...predYs.map(y => y.data.toFixed(3))].join(", ")}] `);
console.log(`Trained Loss: ${loss.data.toFixed(3)}`);


Xs: [[2,3,-1],[3,-1,0.5],[0.5,1,1],[1,1,-1]]Ys: [1,-1,-1,1]
MLP(3, [4, 4, 1]), randomly initialized
Initial Prediction: [-1.812, 0.833, 0.000, -1.010]Initial Loss: 16.308
Training done. Rounds: 100, Learning Rate: 0.01
Trained Prediction: [0.735, -1.021, -0.717, 0.735]Trained Loss: 0.221

undefined